## Example 1: Basic template creation and rendering

In [ ]:
from mako.template import Template
from mako.lookup import TemplateLookup
import time  # used by the status report below

# Example 1: Basic template creation and rendering
print("Example 1: Basic Template Creation and Rendering")

# Create a simple template
basic_template = Template("""
    Hello, ${name}!
    
    You have ${count} message${'' if count == 1 else 's'}.
    
    % if count > 0:
    Your messages:
    % for i, message in enumerate(messages, 1):
        ${i}. ${message}
    % endfor
    % else:
    No messages to display.
    % endif
    
    Best regards,
    ${signature}
    """)

print("Basic template created")

# Render template with different data
users_data = [
    {
        "name": "Alice",
        "count": 3,
        "messages": ["Meeting at 2 PM", "Code review tomorrow", "Project deadline Friday"],
        "signature": "The Team"
    },
    {
        "name": "Bob",
        "count": 1,
        "messages": ["Welcome to the team!"],
        "signature": "HR Department"
    },
    {
        "name": "Carol",
        "count": 0,
        "messages": [],
        "signature": "System Admin"
    }
]

print("\nRendering templates with different data:")

for i, data in enumerate(users_data, 1):
    print(f"\n--- User {i}: {data['name']} ---")
    rendered = basic_template.render(**data)
    print(rendered.strip())

# Template with expressions and formatting
report_template = Template("""
    Sales Report for ${quarter}
    =====================
    
    Total Revenue: $${'{:,.2f}'.format(revenue)}
    Total Orders: ${'{:,}'.format(orders)}
    Average Order: $${'{:.2f}'.format(revenue / orders if orders > 0 else 0)}
    
    % if growth_rate >= 0:
    Growth: +${'{:.1f}%'.format(growth_rate)} 📈
    % else:
    Growth: ${'{:.1f}%'.format(growth_rate)} 📉
    % endif
    
    Top Products:
    % for rank, (product, sales) in enumerate(top_products, 1):
    ${rank}. ${product}: $${'{:,.0f}'.format(sales)}
    % endfor
    """)

# Render sales report
sales_data = {
    "quarter": "Q3 2024",
    "revenue": 1250000.75,
    "orders": 3456,
    "growth_rate": 12.5,
    "top_products": [
        ("Premium Widget", 450000),
        ("Standard Widget", 320000),
        ("Basic Widget", 280000)
    ]
}

print(f"\n--- Sales Report ---")
sales_report = report_template.render(**sales_data)
print(sales_report.strip())
# `${x:,}` is not Mako syntax (a ${} body is a plain Python expression), so the figures are formatted in Python and pinned here.
assert "Total Orders: 3,456" in sales_report and "Total Revenue: $1,250,000.75" in sales_report, sales_report

# Template with Python functions
utility_template = Template("""
    <%!
    import datetime
    import random
    
    def format_timestamp(ts):
        return datetime.datetime.fromtimestamp(ts).strftime('%Y-%m-%d %H:%M:%S')
    
    def random_color():
        colors = ['red', 'blue', 'green', 'yellow', 'purple', 'orange']
        return random.choice(colors)
    %>
    
    System Status Report
    ====================
    Generated: ${format_timestamp(timestamp)}
    
    % for service in services:
    Service: ${service['name']}
    Status: ${service['status'].upper()} (${random_color()})
    Uptime: ${service['uptime']} hours
    
    % endfor
    """)

# System status data
status_data = {
    "timestamp": time.time(),
    "services": [
        {"name": "Web Server", "status": "running", "uptime": 168},
        {"name": "Database", "status": "running", "uptime": 720},
        {"name": "Cache", "status": "degraded", "uptime": 24}
    ]
}

print(f"\n--- System Status ---")
status_report = utility_template.render(**status_data)
print(status_report.strip())

# Show basic template features
print(f"\nBasic template features:")
basic_features = [
    "Variable substitution: ${variable} syntax for clean interpolation",
    "Expression evaluation: Full Python expressions within templates",
    "Control flow: if/else, for loops, and conditional rendering",
    "Built-in functions: Access to Python standard library functions",
    "Formatting support: String formatting and number formatting",
    "Template methods: Define reusable functions within templates"
]

for feature in basic_features:
    print(f"  • {feature}")

print("Basic template creation and rendering completed successfully!")

## Example 2: Template inheritance and includes

In [ ]:
from mako.template import Template
from mako.lookup import TemplateLookup
from mako import exceptions
import tempfile
import os

# Example 2: Template inheritance and includes
print("Example 2: Template Inheritance and Includes")

# Create a base template (layout)
base_template_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>${title} - My Website</title>
        <meta charset="UTF-8">
        <style>
            body { font-family: Arial, sans-serif; margin: 20px; }
            header { background: #f0f0f0; padding: 10px; margin-bottom: 20px; }
            nav { background: #e0e0e0; padding: 5px; margin-bottom: 20px; }
            footer { background: #f0f0f0; padding: 10px; margin-top: 20px; text-align: center; }
            .content { min-height: 400px; }
        </style>
    </head>
    <body>
        <header>
            <h1>My Website</h1>
            <p>Welcome to our site!</p>
        </header>
        
        <nav>
            <%block name="navigation">
            <a href="/">Home</a> | <a href="/about">About</a> | <a href="/contact">Contact</a>
            </%block>
        </nav>
        
        <div class="content">
            <%block name="content">
            <p>Default content goes here.</p>
            </%block>
        </div>
        
        <footer>
            <%block name="footer">
            <p>&copy; 2024 My Website. All rights reserved.</p>
            </%block>
        </footer>
    </body>
    </html>
    """

# Create child templates that inherit from base
home_template_content = """
    <%inherit file="base.html"/>
    
    <%block name="content">
    <h2>Welcome Home!</h2>
    <p>This is the home page with dynamic content.</p>
    
    <h3>Recent News</h3>
    % for article in news:
    <div style="border: 1px solid #ccc; margin: 10px 0; padding: 10px;">
        <h4>${article['title']}</h4>
        <p><em>${article['date']}</em></p>
        <p>${article['summary']}</p>
    </div>
    % endfor
    </%block>
    
    <%block name="footer">
    <p>&copy; 2024 My Website. Last updated: ${last_updated}</p>
    </%block>
    """

about_template_content = """
    <%inherit file="base.html"/>
    
    <%block name="navigation">
    <a href="/">Home</a> | <strong>About</strong> | <a href="/contact">Contact</a>
    </%block>
    
    <%block name="content">
    <h2>About Us</h2>
    <p>We are a company founded in ${founded_year}.</p>
    
    <h3>Our Team</h3>
    % for member in team_members:
    <div style="display: inline-block; margin: 10px; padding: 10px; border: 1px solid #ddd;">
        <h4>${member['name']}</h4>
        <p><strong>${member['role']}</strong></p>
        <p>${member['bio']}</p>
    </div>
    % endfor
    </%block>
    """

# Set up template lookup with temporary directory
with tempfile.TemporaryDirectory() as temp_dir:
    # Write base template to file
    base_file = os.path.join(temp_dir, "base.html")
    with open(base_file, 'w') as f:
        f.write(base_template_content)
    
    # Create lookup object
    lookup = TemplateLookup(directories=[temp_dir])
    
    # Create templates from strings with lookup
    home_template = Template(home_template_content, lookup=lookup)
    about_template = Template(about_template_content, lookup=lookup)
    
    print("Template inheritance system created")
    
    # Data for home page
    home_data = {
        "title": "Home",
        "news": [
            {
                "title": "New Product Launch",
                "date": "2024-01-15",
                "summary": "We're excited to announce our latest product offering!"
            },
            {
                "title": "Company Milestone",
                "date": "2024-01-10", 
                "summary": "We've reached 10,000 satisfied customers!"
            }
        ],
        "last_updated": "2024-01-15 14:30"
    }
    
    # Data for about page
    about_data = {
        "title": "About",
        "founded_year": 2020,
        "team_members": [
            {
                "name": "Alice Johnson",
                "role": "CEO",
                "bio": "Visionary leader with 15 years of experience."
            },
            {
                "name": "Bob Smith",
                "role": "CTO",
                "bio": "Tech expert passionate about innovation."
            }
        ]
    }
    
    print(f"\nRendering home page:")
    home_html = home_template.render(**home_data)
    
    # Extract and show key parts
    content_start = home_html.find("<h2>Welcome Home!")
    content_end = home_html.find("</div>", content_start)
    if content_start != -1 and content_end != -1:
        content = home_html[content_start:content_end]
        print(content[:200] + "..." if len(content) > 200 else content)
    
    print(f"\nRendering about page:")
    about_html = about_template.render(**about_data)
    
    # Extract and show key parts  
    content_start = about_html.find("<h2>About Us")
    content_end = about_html.find("</div>", content_start + 200)  # Look further for team section
    if content_start != -1 and content_end != -1:
        content = about_html[content_start:content_end]
        print(content[:300] + "..." if len(content) > 300 else content)
    
    # Child blocks replace the parent's blocks; blocks a child does not override keep the parent's content
    assert "<title>Home - My Website</title>" in home_html and "<h2>Welcome Home!</h2>" in home_html
    assert "<h4>Company Milestone</h4>" in home_html and "Last updated: 2024-01-15 14:30" in home_html
    assert '<a href="/about">About</a>' in home_html  # inherited navigation block
    assert "<strong>About</strong>" in about_html and "founded in 2020" in about_html
    assert "All rights reserved." in about_html  # inherited footer block

# Template with includes/namespaces
main_template_content = """
    <%namespace name="utils" file="utils.mako"/>
    <%namespace name="widgets" file="widgets.mako"/>
    
    <h1>Dashboard</h1>
    
    ${utils.format_greeting(user_name)}
    
    <div class="metrics">
        ${widgets.metric_card("Users", user_count, "👥")}
        ${widgets.metric_card("Revenue", revenue, "💰")}
        ${widgets.metric_card("Orders", order_count, "📦")}
    </div>
    
    <h2>Recent Activity</h2>
    % for activity in activities:
    ${widgets.activity_item(activity)}
    % endfor
    """

utils_template_content = """
    <%!
    import datetime
    %>
    
    <%def name="format_greeting(name)">
    <%
        hour = datetime.datetime.now().hour
        if hour < 12:
            greeting = "Good morning"
        elif hour < 18:
            greeting = "Good afternoon"
        else:
            greeting = "Good evening"
    %>
    <p style="font-size: 18px; color: #333;">${greeting}, ${name}!</p>
    </%def>
    
    <%def name="format_currency(amount)">
    $${'{:,.2f}'.format(float(amount))}</%def>
    """

widgets_template_content = """
    <%def name="metric_card(title, value, icon)">
    <div style="display: inline-block; margin: 10px; padding: 15px; border: 2px solid #ddd; border-radius: 5px; text-align: center; min-width: 120px;">
        <div style="font-size: 24px;">${icon}</div>
        <h3 style="margin: 5px 0;">${title}</h3>
        <p style="font-size: 20px; font-weight: bold; margin: 5px 0;">
        % if isinstance(value, (int, float)) and title == "Revenue":
        $${'{:,.0f}'.format(value)}
        % elif isinstance(value, (int, float)):
        ${'{:,}'.format(value)}
        % else:
        ${value}
        % endif
        </p>
    </div>
    </%def>
    
    <%def name="activity_item(activity)">
    <div style="padding: 8px; margin: 5px 0; background: #f9f9f9; border-left: 3px solid #007bff;">
        <strong>${activity['user']}</strong> ${activity['action']} 
        <em style="color: #666;">${activity['time']}</em>
    </div>
    </%def>
    """

# Test namespace includes
with tempfile.TemporaryDirectory() as temp_dir:
    # Write utility templates
    utils_file = os.path.join(temp_dir, "utils.mako")
    widgets_file = os.path.join(temp_dir, "widgets.mako")
    
    with open(utils_file, 'w') as f:
        f.write(utils_template_content)
    with open(widgets_file, 'w') as f:
        f.write(widgets_template_content)
    
    # Create lookup and main template
    lookup = TemplateLookup(directories=[temp_dir])
    dashboard_template = Template(main_template_content, lookup=lookup)
    
    # Dashboard data
    dashboard_data = {
        "user_name": "Alice",
        "user_count": 1250,
        "revenue": 45600,
        "order_count": 789,
        "activities": [
            {"user": "Bob", "action": "completed order #1234", "time": "2 minutes ago"},
            {"user": "Carol", "action": "registered new account", "time": "5 minutes ago"},
            {"user": "Dave", "action": "updated profile", "time": "10 minutes ago"}
        ]
    }
    
    print(f"\nRendering dashboard with includes:")
    dashboard_html = dashboard_template.render(**dashboard_data)
    print(dashboard_html[:500] + "..." if len(dashboard_html) > 500 else dashboard_html)
    assert ", Alice!</p>" in dashboard_html and "$45,600" in dashboard_html and "1,250" in dashboard_html
    assert dashboard_html.count("border-left: 3px solid") == 3  # one activity_item per activity

# Show inheritance features
print(f"\nTemplate inheritance features:")
inheritance_features = [
    "Template inheritance: Child templates extend parent templates",
    "Block system: Define replaceable sections in parent templates",
    "Namespace includes: Import and use functions from other templates",
    "Template lookup: Organize templates in directory structures",
    "Code reuse: Share common functionality across multiple templates",
    "Layout management: Consistent layouts with customizable content"
]

for feature in inheritance_features:
    print(f"  • {feature}")

print("Template inheritance and includes completed successfully!")

## Example 3: Filters and formatting

In [ ]:
from mako.template import Template
from mako.runtime import Context
from mako import filters
import io

# Example 3: Filters and formatting
print("Example 3: Filters and Formatting")

# Template with built-in filters
filter_template = Template("""
    User Input Processing Demo
    ========================
    
    Raw HTML content: ${content | n}
    HTML escaped (default filter): ${content}
    URL encoded: ${url_param | n, u}
    XML escaped: ${xml_content | n, x}
    
    Text formatting:
    Uppercase: ${message | upper}
    Lowercase: ${message | lower}
    Trimmed: "${message | trim}"
    
    Number formatting:
    Currency: ${amount | n, currency}
    Percentage: ${rate | n, percentage}
    Decimal places: ${value | n, decimal_format}
    
    List operations:
    Joined: ${tags | n, join_list}
    Sorted: ${numbers | n, sort_numbers}
    """, default_filters=['h'])  # default filters run first on every expression; `n` turns them off

# Define custom filters
def currency_filter(value):
    """Format as currency"""
    return f"${float(value):,.2f}"

def percentage_filter(value):
    """Format as percentage"""  
    return f"{float(value) * 100:.1f}%"

def decimal_format_filter(value):
    """Format with specific decimal places"""
    return f"{float(value):.3f}"

def join_list_filter(items):
    """Join list items with commas"""
    return ", ".join(str(item) for item in items)

def sort_numbers_filter(numbers):
    """Sort numeric list"""
    return str(sorted(numbers))  # a filter's result is written to the output buffer as-is: it must be a str

# Test data with potentially dangerous content
filter_data = {
    # `upper` / `lower` are NOT Mako built-in filters (those are h, u, x, trim, entity, n, ...): a filter name that is
    # not built in is looked up in the render context, so it is supplied here like the custom filters below.
    'upper': str.upper,
    'lower': str.lower,
    'content': '<script>alert("XSS attempt!")</script><b>Bold text</b>',
    'url_param': 'search query with spaces & symbols',
    'xml_content': '<data value="test & production">Content</data>',
    'message': '  Hello World!  ',
    'amount': 12345.678,
    'rate': 0.1545,
    'value': 3.14159265359,
    'tags': ['python', 'mako', 'templates', 'web'],
    'numbers': [42, 7, 23, 91, 15],
    'currency': currency_filter,
    'percentage': percentage_filter,
    'decimal_format': decimal_format_filter,
    'join_list': join_list_filter,
    'sort_numbers': sort_numbers_filter
}

print("Rendering template with filters:")
filtered_output = filter_template.render(**filter_data)
print(filtered_output)
assert "HTML escaped (default filter): &lt;script&gt;alert(" in filtered_output, filtered_output
assert 'Raw HTML content: <script>alert("XSS attempt!")</script>' in filtered_output, filtered_output
assert "URL encoded: search+query+with+spaces+%26+symbols" in filtered_output, filtered_output
assert "XML escaped: &lt;data value=" in filtered_output and "test &amp; production" in filtered_output, filtered_output
assert "HELLO WORLD!" in filtered_output and 'Trimmed: "Hello World!"' in filtered_output, filtered_output
# `n` turns off the default filters, so the list filters receive the list itself rather than an escaped string.
assert "Joined: python, mako, templates, web" in filtered_output, filtered_output
assert "$12,345.68" in filtered_output and "15.4%" in filtered_output and "[7, 15, 23, 42, 91]" in filtered_output, filtered_output

print("✅ Filters and formatting completed successfully!")

## Example 4: Error handling and debugging

In [ ]:
from mako.template import Template
from mako.lookup import TemplateLookup
from mako import exceptions
import time

# Example 4: Error handling and debugging
print("Example 4: Error Handling and Debugging")

# Template that fails at render time (attribute access on a dict)
error_template_content = """
    Template with Errors
    ==================
    
    % for user in users:
    User: ${user.name}
    Email: ${user.email}
    % endfor
    
    Undefined variable: ${undefined_variable}
    """

# Demonstrate error handling
print("Testing error handling mechanisms:")

error_template = Template(error_template_content)
print("✓ Template compiled successfully")

# Test with data that will cause runtime error
error_data = {"users": [{"name": "Alice"}]}  # dicts have no .name attribute
try:
    output = error_template.render(**error_data)
except AttributeError as e:
    # `${user.name}` on a dict: Mako evaluates it as plain Python attribute access
    assert "has no attribute 'name'" in str(e), f"unexpected message: {e!r}"
    print(f"✓ Caught the expected render-time failure ({type(e).__name__}, {e})")
else:
    raise AssertionError("render() should have failed due to missing attribute")

# A malformed template fails when it is compiled: here a `% for` block without `% endfor`
try:
    Template("% for item in items:\n${item}\n")
except exceptions.SyntaxException as e:
    assert "Unterminated control keyword: 'for'" in str(e), f"unexpected message: {e!r}"
    print(f"✓ Caught syntax error: {e}")
    print(f"  Line: {e.lineno}")
else:
    raise AssertionError("Template() should have raised SyntaxException for the missing % endfor")

# Template with proper error handling
safe_template = Template("""
    Safe Template Operations
    ======================
    
    <%def name="safe_divide(a, b)">
    % try:
        ${a / b}
    % except ZeroDivisionError:
        ∞ (division by zero)
    % endtry
    </%def>
    
    <%def name="safe_access(obj, attr, default='N/A')">
    ${getattr(obj, attr, default)}
    </%def>
    
    Safe operations:
    Division: 10 ÷ 2 = ${safe_divide(10, 2)}
    Division by zero: 10 ÷ 0 = ${safe_divide(10, 0)}
    Object access: ${safe_access(user, 'email', 'No email')}
    Missing variable: ${locals().get('missing_var', 'Not provided')}
    """)

# Safe template test data
safe_data = {
    "user": type('User', (), {"name": "Bob"})(),  # Object without email
}

print(f"\nSafe operations output:")
safe_output = safe_template.render(**safe_data)
print(safe_output.strip())
# One `% except` per control-line `try`: Mako's PythonPrinter does not dedent a second `% except`.
assert "5.0" in safe_output and "∞ (division by zero)" in safe_output and "No email" in safe_output, safe_output

print("✅ Error handling and debugging completed successfully!")

## Example 5: Performance optimization and caching

In [ ]:
from mako.template import Template
from mako.lookup import TemplateLookup
from mako import cache
import time
import tempfile
import os

# Example 5: Performance optimization and caching
print("Example 5: Performance Optimization and Caching")

# Performance testing setup
def create_test_template():
    """Create a complex template for performance testing"""
    return Template("""
        <%!
        import datetime
        %>
        
        Performance Test Template
        =======================
        Generated: ${datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
        
        % for section in sections:
        ${"##"} ${section['title']}
        
        % for i, item in enumerate(section['items']):
        ${i + 1}. **${item['name']}** (Priority: ${item['priority']})
           Description: ${item['description']}
           % if item.get('details'):
           Details:
           % for detail in item['details']:
           - ${detail}
           % endfor
           % endif
           
        % endfor
        
        Summary: ${len(section['items'])} items in ${section['title']}
        % endfor
        
        Total sections: ${len(sections)}
        Total items: ${sum(len(s['items']) for s in sections)}
        """)

# Generate test data
def generate_test_data(num_sections=10, items_per_section=20):
    """Generate complex test data"""
    sections = []
    
    for s in range(num_sections):
        items = []
        for i in range(items_per_section):
            item = {
                'name': f'Item {s}-{i}',
                'priority': ['High', 'Medium', 'Low'][i % 3],
                'description': f'This is description for item {i} in section {s}',
                'details': [f'Detail {d}' for d in range(i % 5)]
            }
            items.append(item)
        
        section = {
            'title': f'Section {s + 1}',
            'items': items
        }
        sections.append(section)
    
    return {'sections': sections}

# Performance benchmarking
def benchmark_template_rendering(template, data, iterations=100):
    """Benchmark template rendering performance"""
    times = []
    
    for _ in range(iterations):
        start = time.time()
        output = template.render(**data)
        end = time.time()
        times.append(end - start)
    
    avg_time = sum(times) / len(times)
    min_time = min(times)
    max_time = max(times)
    output_size = len(output)
    
    return {
        'avg_time': avg_time,
        'min_time': min_time,
        'max_time': max_time,
        'output_size': output_size,
        'throughput': 1.0 / avg_time
    }

# Test basic performance
print("Basic performance testing:")

template = create_test_template()
small_data = generate_test_data(5, 10)
large_data = generate_test_data(20, 50)

# Small data benchmark
small_output = template.render(**small_data)
# A line starting with `##` is a Mako comment, so the heading marker is emitted as an expression
assert all(f"## {section['title']}" in small_output for section in small_data['sections']), small_output[:500]
assert "Total sections: 5" in small_output and "Total items: 50" in small_output

small_bench = benchmark_template_rendering(template, small_data, 50)
print(f"Small dataset (5 sections, 10 items each):")
print(f"  Average time: {small_bench['avg_time']*1000:.2f}ms")
print(f"  Throughput: {small_bench['throughput']:.1f} renders/sec")
print(f"  Output size: {small_bench['output_size']:,} characters")

# Large data benchmark
large_bench = benchmark_template_rendering(template, large_data, 20)
print(f"\nLarge dataset (20 sections, 50 items each):")
print(f"  Average time: {large_bench['avg_time']*1000:.2f}ms")
print(f"  Throughput: {large_bench['throughput']:.1f} renders/sec")
print(f"  Output size: {large_bench['output_size']:,} characters")

# Template caching with TemplateLookup
with tempfile.TemporaryDirectory() as temp_dir:
    # Create cached lookup
    lookup = TemplateLookup(
        directories=[temp_dir],
        module_directory=os.path.join(temp_dir, 'cache'),
        filesystem_checks=False  # Disable file change checks for performance
    )
    
    # Create template file
    template_file = os.path.join(temp_dir, 'cached_template.mako')
    with open(template_file, 'w') as f:
        f.write("""
            <%!
            import datetime
            %>
            
            Cached Template Performance Test
            ==============================
            Generated: ${datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
            Template compilation time: ${compile_time}
            
            % for i in range(iterations):
            Iteration ${i + 1}: ${data_value * (i + 1)}
            % endfor
            
            Performance summary:
            - Template cached: Yes
            - Iterations: ${iterations}
            - Base value: ${data_value}
            """)
    
    print(f"\nCached template performance:")
    
    # Test cached template loading and rendering
    cache_data = {'compile_time': 'Pre-compiled', 'iterations': 100, 'data_value': 42}
    
    # First load (with compilation)
    start = time.time()
    cached_template = lookup.get_template('cached_template.mako')
    compilation_time = time.time() - start
    
    # First render
    start = time.time()
    first_output = cached_template.render(**cache_data)
    first_render_time = time.time() - start
    
    # Subsequent renders (using cached compiled template)
    render_times = []
    for _ in range(10):
        start = time.time()
        output = cached_template.render(**cache_data)
        render_times.append(time.time() - start)
    
    avg_cached_time = sum(render_times) / len(render_times)
    
    print(f"  Compilation time: {compilation_time*1000:.2f}ms")
    print(f"  First render: {first_render_time*1000:.2f}ms")
    print(f"  Cached renders average: {avg_cached_time*1000:.2f}ms")
    print(f"  Performance improvement: {first_render_time/avg_cached_time:.2f}x")

# Memory-based caching
class MemoryCache:
    """Simple memory-based template cache"""
    def __init__(self):
        self.cache = {}
        self.stats = {'hits': 0, 'misses': 0}
    
    def get_template(self, template_content):
        key = hash(template_content)
        
        if key in self.cache:
            self.stats['hits'] += 1
            return self.cache[key]
        
        self.stats['misses'] += 1
        template = Template(template_content)
        self.cache[key] = template
        return template
    
    def get_stats(self):
        return self.stats.copy()

# Test memory caching
print(f"\nMemory cache performance:")

mem_cache = MemoryCache()
simple_template_content = """
    Simple template ${value}
    % for i in range(count):
    Item ${i}: ${value * i}
    % endfor
    """

# Test cache performance
cache_test_data = {'value': 10, 'count': 50}
cache_render_times = []

for i in range(20):
    start = time.time()
    template = mem_cache.get_template(simple_template_content)
    output = template.render(**cache_test_data)
    cache_render_times.append(time.time() - start)

cache_stats = mem_cache.get_stats()
avg_cache_time = sum(cache_render_times) / len(cache_render_times)

print(f"  Cache hits: {cache_stats['hits']}")
print(f"  Cache misses: {cache_stats['misses']}")
print(f"  Average time: {avg_cache_time*1000:.2f}ms")
print(f"  Hit rate: {cache_stats['hits']/(cache_stats['hits'] + cache_stats['misses'])*100:.1f}%")

# Template optimization techniques
print(f"\nTemplate optimization techniques:")

# Optimized template with pre-computed values
optimized_template = Template("""
    <%!
    # Pre-compute expensive operations
    def expensive_calculation(n):
        return sum(i * i for i in range(n))
    %>
    
    Optimized Template
    ================
    
    % for item in items:
    Item: ${item['name']}
    ## .get() supplies a default for a missing key
    Value: ${item.get('value', 0)}
    % if item.get('compute_expensive'):
    Expensive result: ${expensive_calculation(item['size'])}
    % endif
    % endfor
    """)

# Compare with non-optimized template
non_optimized_template = Template("""
    Non-Optimized Template
    =====================
    
    % for item in items:
    Item: ${item['name']}
    ## membership test before indexing
    Value: ${item['value'] if 'value' in item else 0}
    % if 'compute_expensive' in item and item['compute_expensive']:
    ## computed inline in the expression
    Expensive result: ${sum(i * i for i in range(item['size']))}
    % endif
    % endfor
    """)

# Optimization test data
opt_data = {
    'items': [
        {'name': f'Item {i}', 'value': i * 10, 'size': 100, 'compute_expensive': i % 3 == 0}
        for i in range(20)
    ]
}

# Both templates render the same item lines; `##` lines are template comments and do not appear
opt_output = optimized_template.render(**opt_data)
non_opt_output = non_optimized_template.render(**opt_data)
item_lines = lambda text: [line.strip() for line in text.splitlines() if line.strip().startswith(("Item:", "Value:", "Expensive"))]
assert item_lines(opt_output) == item_lines(non_opt_output), (opt_output, non_opt_output)
assert item_lines(opt_output).count("Expensive result: 328350") == 7  # items 0, 3, ..., 18
assert "#" not in opt_output and "#" not in non_opt_output

# Benchmark both templates
opt_bench = benchmark_template_rendering(optimized_template, opt_data, 50)
non_opt_bench = benchmark_template_rendering(non_optimized_template, opt_data, 50)

print(f"  Optimized template: {opt_bench['avg_time']*1000:.2f}ms")
print(f"  Non-optimized template: {non_opt_bench['avg_time']*1000:.2f}ms")
print(f"  Performance improvement: {non_opt_bench['avg_time']/opt_bench['avg_time']:.2f}x")

# Production optimization strategies
print(f"\nProduction optimization strategies:")
optimization_strategies = [
    "Template caching: Pre-compile templates and cache for reuse",
    "Filesystem checks: Disable file modification checks in production",
    "Memory optimization: Use template lookup directories for organization",
    "Pre-computation: Move expensive calculations to template functions",
    "Safe access: Use .get() methods for optional dictionary keys",
    "Batch rendering: Process multiple items efficiently in loops",
    "Output caching: Cache rendered output for frequently used data"
]

for strategy in optimization_strategies:
    print(f"  • {strategy}")

# Show performance benefits summary
print(f"\nPerformance benefits achieved:")
benefits = [
    f"Caching speedup: {first_render_time/avg_cached_time:.2f}x faster for repeated renders",
    f"Template optimization: {non_opt_bench['avg_time']/opt_bench['avg_time']:.2f}x faster execution",
    f"Memory efficiency: {cache_stats['hits']}/{cache_stats['hits'] + cache_stats['misses']} cache hit rate",
    f"Throughput scaling: {large_bench['throughput']:.1f} renders/sec for large datasets"
]

for benefit in benefits:
    print(f"  • {benefit}")

print("Performance optimization and caching completed successfully!")